# Corrupt & probe EntailmentBank chains

1. **Load** EntailmentBank as `ReasoningChain`s and inspect them one by one.
2. **Pick** one or more chains and **corrupt** them however you like.
3. **Run an NNsight model** over the chosen chains (valid + corrupted) and read
   out the curve-geometry measurements *and* the raw hidden states at
   `L/3` and `2L/3` of the network depth.

Needs: `torch`, `nnsight`, `transformers` (plus `pandas` / `matplotlib` /
`scikit-learn`). The EntailmentBank v3 release must be unzipped at
`../data/raw/entailment_trees_emnlp2021_data_v3/`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from reasoning_trajectories import Config, ReasoningChain
from reasoning_trajectories.reasoning import CORRUPTIONS, DonorPool
from reasoning_trajectories.datasets.entailmentbank import EntailmentBank
from reasoning_trajectories.extraction import Extractor, ReasoningModel, layers as layer_utils
from reasoning_trajectories.trajectories import Guards, curvature, summarize_trajectory

REPO = Path.cwd().parent
EB_ROOT = REPO / "data" / "raw" / "entailment_trees_emnlp2021_data_v3"
pd.set_option("display.max_colwidth", 140)
print("corruption kinds:", list(CORRUPTIONS))

## 1. Load & browse the chains

`ReasoningChain.from_example` gives each gold tree as an object with `.text`,
`.steps`, `.premises`, `.hypothesis`, `.leaves`, `.intermediates`,
`.as_format(...)`, and the corruption methods.

In [ ]:
TASK = 1               # 1 | 2 | 3
SPLIT = "validation"   # train | validation | test
FMT = "natural"        # natural | structured | raw

examples = list(EntailmentBank(task=TASK, split=SPLIT, local_dir=EB_ROOT, shuffle_seed=0))
chains = {e.id: ReasoningChain.from_example(e, fmt=FMT) for e in examples}
by_id = {e.id: e for e in examples}
donors = DonorPool(chains.values())          # cross-tree sentence pool for substitutions

overview = pd.DataFrame([{
    "i": i,
    "id": e.id,
    "n_steps": len(chains[e.id]),
    "depth": e.meta.get("depth_of_proof"),
    "n_leaves": len(chains[e.id].leaves),
    "chain_words": len(chains[e.id].text.split()),
    "hypothesis": chains[e.id].hypothesis,
} for i, e in enumerate(examples)])
print(len(examples), "chains loaded")
overview.head(25)

In [ ]:
def show_chain(key, fmt=None):
    """Inspect one chain by row index or id."""
    ex = by_id[key] if key in by_id else examples[key]
    ch = chains[ex.id] if fmt is None else ReasoningChain.from_example(ex, fmt=fmt)
    print("id:", ex.id, "| steps:", len(ch), "| depth:", ex.meta.get("depth_of_proof"))
    print("\nPROMPT (fed to the model before the chain):")
    print("-" * 70)
    print(ex.prompt)
    print("\nCHAIN TEXT (meta['reasoning']):")
    print("-" * 70)
    print(ch.text)
    print("\nCOMPONENTS:")
    print("  hypothesis   :", ch.hypothesis)
    print("  leaves       :", ch.leaves)
    print("  intermediates:", ch.intermediates)
    print("  steps        :")
    for s in ch.steps:
        print("     ", list(s.premise_ids), "->", s.conclusion_id + ":", s.conclusion)
    return ch

_ = show_chain(0)

## 2. Pick chains & corrupt them

Edit `SELECT` and `KINDS`. Each selected chain becomes a `valid` variant plus
one variant per corruption kind. `.corrupt()` returns a **new** chain; the
change record is on `.meta['corruption']`.

| kind | what it does |
|---|---|
| `step_shuffle` | reorder steps (non-identity); chains with < `MIN_STEPS` steps escalate to `premise_substitution` |
| `premise_substitution` | swap one premise for another sentence **from the same tree** |
| `premise_substitution_other_tree` | swap one premise for a sentence from a **different** tree |
| `conclusion_substitution` | swap one step's conclusion for another tree's; propagates to later steps that used it |

In [ ]:
# ------------------------------- EDIT -------------------------------
# Row indices (see the overview table) or ids. For step_shuffle to actually
# reorder, pick rows with n_steps >= 3 (shorter chains escalate to a premise
# swap). The <= 200 word filter keeps prompt+chain inside gpt2's 1024 window.
SELECT    = list(overview.query("n_steps >= 3 and chain_words <= 200")["i"][:2])
KINDS     = ["step_shuffle", "premise_substitution_other_tree", "conclusion_substitution"]
SEED      = 0
MIN_STEPS = 3
# ------------------------------------------------------------------

def _id(key):
    return key if key in by_id else examples[key].id

def variants_of(cid):
    base = chains[cid]
    out = {"valid": base}
    for k in KINDS:
        out[k] = base.corrupt(k, seed=SEED, donors=donors, min_steps=MIN_STEPS)
    return out

selected_ids = [_id(k) for k in SELECT]
variants = {cid: variants_of(cid) for cid in selected_ids}

for cid, vs in variants.items():
    print("=" * 88)
    print(cid)
    for name, ch in vs.items():
        info = ch.meta.get("corruption")
        tag = {k: v for k, v in info.items() if k != "permutation"} if info else ""
        print("\n---", name, "---  ", tag)
        print(ch.text)

## 3. Run an NNsight model

Set `MODEL_NAME` / `REMOTE`. `openai-community/gpt2` runs locally with no
download gate and is fine for a quick check; swap in
`meta-llama/Meta-Llama-3.1-8B` (local, gated) or set `REMOTE = True` to run it
on NDIF (needs `NDIF_API_KEY`).

In [ ]:
cfg = Config.load(str(REPO / "configs" / "entailmentbank.yaml"))

MODEL_NAME = "openai-community/gpt2"     # or cfg.model.name, or a DeepSeek-R1 distill on NDIF
REMOTE = False

cfg.model.name = MODEL_NAME
cfg.model.remote = REMOTE
if "gpt2" in MODEL_NAME:
    cfg.model.dtype = "float32"
    cfg.model.device_map = "cpu"

model = ReasoningModel(cfg.model)
L = model.n_layers
L3 = layer_utils.resolve_layer(1 / 3, L)     # residual stream at ~L/3
L23 = layer_utils.resolve_layer(2 / 3, L)    # residual stream at ~2L/3  (the note's l*)
print(MODEL_NAME, " L =", L, "blocks, hidden =", model.hidden_size)
print("probing residual-stream layers  L/3 =", L3, "  2L/3 =", L23)

# target exactly those two layers for the token trajectory; layer trajectory uses all L+1
cfg.extraction.star_layer = L23
cfg.extraction.extra_token_layers = [L3]
cfg.extraction.chain_span = "completion"     # hidden states over the chain tokens only
extractor = Extractor(model, cfg.extraction)
guards = Guards.from_config(cfg.geometry)

In [ ]:
def probe(ex, chain):
    """One trace over  ex.prompt + chain.text  ->  geometry + raw hidden states."""
    rec = extractor.extract(ex, chain.text, predicted=None)
    feats = {"layer_trajectory": summarize_trajectory(rec.layer_trajectory.points, guards)}
    hidden = {}
    for l, tt in rec.token_trajectories.items():
        where = "2L/3" if l == L23 else "L/3"
        feats["token@" + str(l) + " (" + where + ")"] = summarize_trajectory(tt.points, guards)
        hidden[l] = tt.points                        # (n_chain_tokens, hidden_dim), float64
    return {
        "tokens": rec.tokens,                         # chain tokens (strings)
        "n_chain_tokens": len(rec.tokens),
        "features": feats,
        "hidden": hidden,                             # {L3: array, L23: array}
        "layer_means": rec.layer_trajectory.points,   # (L+1, hidden_dim)
    }

In [ ]:
RESULTS = {}
rows = []
for cid, vs in variants.items():
    ex = by_id[cid]
    RESULTS[cid] = {}
    for name, ch in vs.items():
        r = probe(ex, ch)
        RESULTS[cid][name] = r
        for traj, f in r["features"].items():
            rows.append({"id": cid, "variant": name, "trajectory": traj,
                         "n_chain_tokens": r["n_chain_tokens"], **f})
        print(cid, name, " tokens =", r["n_chain_tokens"])

feat_df = pd.DataFrame(rows)
feat_df

### The measurements

`feat_df` has one row per (chain, variant, trajectory) with `path_length` (S),
`dirichlet_energy` (E), `straightness` (sigma), and the mean / max / std of
curvature and torsion, plus the guard-excluded fractions.

In [ ]:
# corrupted - valid delta, per chain, for the 2L/3 token trajectory
tok = feat_df[feat_df["trajectory"].str.contains("2L/3")].set_index(["id", "variant"])
cols = ["path_length", "dirichlet_energy", "straightness",
        "curvature_mean", "curvature_max", "torsion_mean", "torsion_max"]
deltas = []
for cid in selected_ids:
    base = tok.loc[(cid, "valid"), cols]
    for name in KINDS:
        if (cid, name) in tok.index:
            deltas.append({"id": cid, "variant": name,
                           **(tok.loc[(cid, name), cols] - base).to_dict()})
pd.DataFrame(deltas).round(3)

### The hidden states themselves

`RESULTS[chain_id][variant]["hidden"][layer]` is the residual-stream hidden
state of every chain token at that layer -- shape `(n_chain_tokens, hidden_dim)`.

In [ ]:
cid = selected_ids[0]
for name in RESULTS[cid]:
    h3 = RESULTS[cid][name]["hidden"][L3]
    h23 = RESULTS[cid][name]["hidden"][L23]
    print(name, "  L/3", h3.shape, "  2L/3", h23.shape, "  dtype", h23.dtype)

# the 2L/3 hidden state of the first chain token, valid variant (first 8 dims):
RESULTS[cid]["valid"]["hidden"][L23][0][:8]

In [ ]:
# optional: dump every array to disk
OUT = Path.cwd() / "probe_output"
OUT.mkdir(exist_ok=True)
for cid, vs in RESULTS.items():
    for name, r in vs.items():
        np.savez_compressed(
            OUT / (cid + "__" + name + ".npz"),
            hidden_L3=r["hidden"][L3].astype(np.float16),
            hidden_L23=r["hidden"][L23].astype(np.float16),
            layer_means=r["layer_means"].astype(np.float32),
            tokens=np.array(r["tokens"], dtype=object),
        )
feat_df.to_csv(OUT / "features.csv", index=False)
print("wrote", len(list(OUT.glob("*.npz"))), "npz +", OUT / "features.csv")

### Visualise one chain: valid vs. corrupted

In [ ]:
from sklearn.decomposition import PCA

cid = selected_ids[0]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, r in RESULTS[cid].items():
    X = r["hidden"][L23].astype(float)
    k, kv = curvature(X, guards)
    axes[0].plot(np.where(kv, k, np.nan), label=name, alpha=0.8)
    Y = PCA(n_components=2).fit_transform(X)
    axes[1].plot(Y[:, 0], Y[:, 1], "-o", ms=3, lw=1, label=name, alpha=0.7)
axes[0].set_title(cid + ": curvature along token trajectory @ 2L/3")
axes[0].set_xlabel("interior vertex")
axes[0].legend(fontsize=7)
axes[1].set_title("PCA of the token trajectory (visualisation only)")
axes[1].legend(fontsize=7)
fig.tight_layout()

---
**Notes**

- gpt2 has a 1024-token context; the default `SELECT` filters to
  `chain_words <= 200`. Use a larger-window model for deeper chains.
- `Extractor` re-tokenises `prompt + chain.text` as one string; the
  prompt/chain boundary can shift by a token vs. tokenising them separately.
- Feeding a corrupted chain is off-policy (the model would not generate it) --
  the geometry describes how the model *encodes* that text.
- To sweep the whole split instead of hand-picked chains, use
  `configs/entailmentbank_corruptions.yaml` with `scripts/run_extraction.py`.